In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import json
import pickle
import pandas as pd

In [2]:
load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)

In [3]:
import pickle
import pandas as pd
import json
with open("logreg_model.pkl", "rb") as f:
    logreg_bundle = pickle.load(f)

logreg_model = logreg_bundle["model"]
logreg_features = logreg_bundle["features"]
logreg_threshold = logreg_bundle["threshold"]

with open("gridsearch_crash_model.pkl", "rb") as f:
    gridsearch_model = pickle.load(f)

with open("sarimax_model.pkl", "rb") as f:
    sarimax_model = pickle.load(f)

with open("model_features.pkl", "rb") as f:
    model_features = pickle.load(f)
with open("clean_crash_data.pkl","rb") as f:
    main_df =  pickle.load(f)


In [4]:
main_df = main_df[main_df["BOROUGH"] != "UNKNOWN"]

In [5]:

borough_stats_df = main_df.groupby("BOROUGH").agg(
    total_crashes=("BOROUGH", "size"),
    total_injuries=("NUMBER OF PERSONS INJURED", "sum")
).reset_index()

borough_stats_df["injury_rate"] = (
    borough_stats_df["total_injuries"] / borough_stats_df["total_crashes"]
)

borough_stats_df = borough_stats_df.sort_values("injury_rate", ascending=False)

In [6]:
def get_borough_statistics(comparison_boroughs=None):
    global borough_stats_df

    df = borough_stats_df.copy()

    # comparison varsa
    if comparison_boroughs:
        wanted = [b.lower() for b in comparison_boroughs]
        df = df[df["BOROUGH"].str.lower().isin(wanted)]

        return {
            "type": "borough_comparison",
            "borough_stats": df.to_dict(orient="records")
        }

    # genel analiz
    highest = df.iloc[0]
    safest = df.sort_values("injury_rate", ascending=True).iloc[0]
    most_crashes = df.sort_values("total_crashes", ascending=False).iloc[0]

    return {
        "type": "borough_overview",
        "highest_injury_rate_borough": highest.to_dict(),
        "safest_borough": safest.to_dict(),
        "most_crashes_borough": most_crashes.to_dict(),
        "borough_stats": df.to_dict(orient="records")
    }

In [7]:
def prepare_input(borough,hour,day_of_week):
    main_df=pd.DataFrame([{"borough":borough,
                           "hour":hour,
                           "day_of_week":day_of_week}])
    main_df = pd.get_dummies(main_df)
    main_df = main_df.reindex(columns=model_features,fill_value=0)
    return main_df

In [8]:
def predict_with_logreg(borough, hour, day_of_week):
    X = prepare_input(borough, hour, day_of_week)

    proba = logreg_model.predict_proba(X)[0][1]
    pred = 1 if proba >= logreg_threshold else 0

    return {
        "model": "logreg",
        "prediction": int(pred),
        "probability": float(proba),
        "threshold": float(logreg_threshold)
    }

In [9]:
def predict_with_gridsearch(borough, hour, day_of_week):
    X = prepare_input(borough, hour, day_of_week)
    pred = gridsearch_model.predict(X)[0]

    result = {"model": "gridsearch", "prediction": int(pred)}

    if hasattr(gridsearch_model, "predict_proba"):
        proba = gridsearch_model.predict_proba(X)[0].tolist()
        result["probabilities"] = proba

    return result

In [10]:
def forecast_with_sarimax(steps=24):
    forecast = sarimax_model.forecast(steps=steps)
    
    if hasattr(forecast, "tolist"):
        forecast_values = forecast.tolist()
    else:
        forecast_values = list(forecast)

    return {
        "model": "sarimax",
        "steps": steps,
        "forecast": forecast_values
    }


In [11]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "predict_with_logreg",
            "description": "Predict injury risk for a given NYC crash scenario using logistic regression (scenario-based prediction, not historical data).",
            "parameters": {
                "type": "object",
                "properties": {
                    "borough": {
                        "type": "string",
                        "description": "NYC borough name (e.g., Brooklyn, Queens, Manhattan, Bronx, Staten Island)"
                    },
                    "hour": {
                        "type": "integer",
                        "description": "Hour of day (0–23)"
                    },
                    "day_of_week": {
                        "type": "integer",
                        "description": "Day of week (0=Monday, 6=Sunday)"
                    }
                },
                "required": ["borough", "hour", "day_of_week"],
                "additionalProperties": False
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "predict_with_gridsearch",
            "description": "Predict injury risk using the optimized classification model (GridSearch). Scenario-based prediction.",
            "parameters": {
                "type": "object",
                "properties": {
                    "borough": {
                        "type": "string",
                        "description": "NYC borough name"
                    },
                    "hour": {
                        "type": "integer",
                        "description": "Hour of day (0–23)"
                    },
                    "day_of_week": {
                        "type": "integer",
                        "description": "Day of week (0–6)"
                    }
                },
                "required": ["borough", "hour", "day_of_week"],
                "additionalProperties": False
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "forecast_with_sarimax",
            "description": "Forecast future NYC crash counts using SARIMAX (time-series prediction).",
            "parameters": {
                "type": "object",
                "properties": {
                    "steps": {
                        "type": "integer",
                        "description": "Number of future hours to forecast"
                    }
                },
                "required": ["steps"],
                "additionalProperties": False
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "get_analysis_summary",
            "description": "Return a summary of the NYC crash analysis project, including models and objectives.",
            "parameters": {
                "type": "object",
                "properties": {},
                "additionalProperties": False
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "get_borough_statistics",
            "description": "Return historical borough-level crash statistics including total crashes, total injuries, injury rate, safest borough, and comparisons between boroughs.",
            "parameters": {
                "type": "object",
                "properties": {
                    "comparison_boroughs": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "Optional list of borough names to compare (e.g., ['Brooklyn', 'Queens'])"
                    }
                },
                "additionalProperties": False
            }
        }
    }
]

In [12]:
system_prompt = """
You are an AI assistant specialized in NYC motor vehicle crash analysis.

IMPORTANT:
You only know what is available through the provided tools.
You MUST use tools when needed.
Do not pretend to retrieve or calculate data unless you are calling a tool.

LANGUAGE RULE:
- If the user writes in Turkish, respond in Turkish.
- If the user writes in English, respond in English.

AVAILABLE TOOLS:
1. predict_with_logreg
   - Predicts injury risk for a given scenario (borough, hour, day_of_week)

2. predict_with_gridsearch
   - Predicts injury risk using the optimized classification model

3. forecast_with_sarimax
   - Forecasts future crash counts

4. get_analysis_summary
   - Returns project summary

5. get_borough_statistics
   - Returns historical borough-level crash statistics, injury rates, and comparisons

--------------------------------------------------

TASK TYPES AND WHAT TO DO:

1. HISTORICAL QUESTIONS (VERY IMPORTANT)
If the user asks:
- Which borough has the highest injury rate
- Safest borough
- Compare Brooklyn vs Queens
- Crash frequency by borough
- Injury rate ranking

👉 ALWAYS call: get_borough_statistics

NEVER say it is unsupported.

--------------------------------------------------

2. SCENARIO-BASED PREDICTION
If the user gives:
- borough + hour + day

👉 use:
- predict_with_logreg OR
- predict_with_gridsearch

--------------------------------------------------

3. FORECAST QUESTIONS
If the user asks about:
- future crashes
- next hours / days

👉 use:
- forecast_with_sarimax

--------------------------------------------------

4. PROJECT QUESTIONS
If the user asks about:
- models
- project summary

👉 use:
- get_analysis_summary

--------------------------------------------------

CRITICAL RULES:
- NEVER invent numbers
- NEVER guess borough rankings
- ALWAYS use a tool when data is needed
- NEVER say “not supported” if a relevant tool exists
- Clearly distinguish:
  - historical observed data
  - model prediction

--------------------------------------------------

RESPONSE STYLE:
- Clear and professional
- Short but informative
- If tool result is used, explain it naturally (not raw JSON)
"""

In [13]:
analysis_summary = {
    "project": "NYC crash analysis",
    "focus": "crash patterns, borough comparisons, injury prediction",
    "models": ["logistic regression", "grid search model", "sarimax"]}

In [14]:
conversation_history = []

def chat(user_message):
    global conversation_history

    conversation_history.append({
        "role": "user",
        "content": user_message
    })

    while True:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "system", "content": system_prompt}] + conversation_history,
            tools=tools,
            tool_choice="auto",
            temperature=0.2
        )

        message = response.choices[0].message

        # Tool call varsa
        if getattr(message, "tool_calls", None):
            conversation_history.append(message)

            for tool_call in message.tool_calls:
                fn_name = tool_call.function.name
                args = json.loads(tool_call.function.arguments) if tool_call.function.arguments else {}

                try:
                    if fn_name == "predict_with_logreg":
                        result = predict_with_logreg(**args)

                    elif fn_name == "predict_with_gridsearch":
                        result = predict_with_gridsearch(**args)

                    elif fn_name == "forecast_with_sarimax":
                        result = forecast_with_sarimax(**args)

                    elif fn_name == "get_analysis_summary":
                        result = analysis_summary

                    elif fn_name == "get_borough_statistics":
                        result = get_borough_statistics(**args)

                    else:
                        result = {"error": f"Unknown function: {fn_name}"}

                except Exception as e:
                    result = {"error": str(e)}

                conversation_history.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result)
                })

            continue

        # Normal assistant cevabı
        assistant_message = message.content if message.content else "I could not generate a response."

        conversation_history.append({
            "role": "assistant",
            "content": assistant_message
        })

        return print(assistant_message)

In [15]:
chat("Which borough has the highest injury rate?")

Brooklyn has the highest injury rate among the NYC boroughs, with an injury rate of approximately 35.28%.


In [16]:
chat("Compare Brooklyn and Queens in terms of crash frequency and injury rate.")

Here's a comparison between Brooklyn and Queens in terms of crash frequency and injury rate:

- **Brooklyn**:
  - Total Crashes: 498,993
  - Total Injuries: 176,051
  - Injury Rate: 35.28%

- **Queens**:
  - Total Crashes: 416,524
  - Total Injuries: 133,416
  - Injury Rate: 32.03%

Brooklyn has both a higher number of total crashes and a higher injury rate compared to Queens.


In [17]:
chat("Which borough has the highest injury rate?")

Brooklyn has the highest injury rate among the NYC boroughs, with an injury rate of approximately 35.28%.


In [18]:
chat("average injury rate")

The average injury rate across all boroughs can be calculated by considering the total injuries and total crashes for each borough. Here's the data:

- **Brooklyn**: 35.28%
- **Bronx**: 34.95%
- **Queens**: 32.03%
- **Staten Island**: 30.18%
- **Manhattan**: 22.98%

To find the average injury rate, we can sum these rates and divide by the number of boroughs:

\[
\text{Average Injury Rate} = \frac{35.28 + 34.95 + 32.03 + 30.18 + 22.98}{5} \approx 31.88\%
\]

The average injury rate across all boroughs is approximately 31.88%.


In [19]:
chat("total crash")

The total number of crashes across all NYC boroughs is the sum of crashes from each borough:

- **Brooklyn**: 498,993
- **Bronx**: 230,307
- **Queens**: 416,524
- **Staten Island**: 65,150
- **Manhattan**: 343,626

Adding these together gives:

\[
\text{Total Crashes} = 498,993 + 230,307 + 416,524 + 65,150 + 343,626 = 1,554,600
\]

The total number of crashes across all boroughs is 1,554,600.


In [20]:
chat("hangi yillar arasinda")

Üzgünüm, ancak elimdeki verilere göre kazaların hangi yıllar arasında toplandığı bilgisine sahip değilim. Eğer başka bir konuda yardımcı olabileceğim bir şey varsa lütfen belirtin!


In [21]:
chat("data timeline")

Unfortunately, the specific timeline or years covered by the data used in the NYC crash analysis project is not provided in the summary. The project focuses on analyzing crash patterns, comparing boroughs, and predicting injuries using models like logistic regression, grid search, and SARIMAX. If you have any other questions or need further assistance, feel free to ask!


In [23]:
main_df['CRASH DATE'].min()

Timestamp('2012-07-01 00:00:00')

In [24]:
main_df['CRASH DATE'].max()

Timestamp('2026-03-03 00:00:00')